In [10]:
import os, glob
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications.resnet50 import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix

In [11]:
file_path_train = 'archive/train'
file_path_test = 'archive/test'
classes = os.listdir(file_path_train)
print(classes)

['happy', '.DS_Store', 'sad', 'fear', 'surprise', 'neutral', 'angry', 'disgust']


In [12]:
filepath = list(glob.glob(file_path_train + '/**/*.*'))
labels = list(map(lambda x: os.path.split(os.path.split(x)[0])[1], filepath))


In [13]:
filepath = pd.Series(filepath, name='Filepath').astype(str)
labels = pd.Series(labels, name='Labels')
data = pd.concat([filepath, labels], axis=1)
data = data.sample(frac=1).reset_index(drop=True)  # Shuffle the dataset
print(data.head())


                                      Filepath   Labels
0     archive/train/happy/Training_4382698.jpg    happy
1    archive/train/happy/Training_73600586.jpg    happy
2  archive/train/neutral/Training_67475621.jpg  neutral
3     archive/train/fear/Training_31840472.jpg     fear
4    archive/train/angry/Training_66839977.jpg    angry


In [14]:
count = data['Labels'].value_counts()
min_count = count.min()
balanced_data = []


In [15]:
for label in count.index:
    class_samples = data[data['Labels'] == label]
    if len(class_samples) > min_count:
        balanced_data.append(class_samples.sample(min_count, random_state=92))
    else:
        balanced_data.append(class_samples.sample(min_count, replace=True, random_state=62))

# Concatenate balanced data and shuffle
balanced_data = pd.concat(balanced_data).sample(frac=1, random_state=42).reset_index(drop=True)

In [17]:
Train, Test = train_test_split(balanced_data, test_size=0.2, random_state=132)
print(f"Training shape: {Train.shape}, Testing shape: {Test.shape}")

Training shape: (2441, 2), Testing shape: (611, 2)


In [18]:
train_pre_processing = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

test_pre_processing = ImageDataGenerator(rescale=1./255)

# Create generators for training, validation, and testing data
train_pre_processed = train_pre_processing.flow_from_dataframe(
    dataframe=Train,
    x_col='Filepath',
    y_col='Labels',
    target_size=(48, 48),
    class_mode='categorical',
    color_mode='grayscale',
    batch_size=32,
    shuffle=True,
    seed=42
)

Validation_pre_processed = test_pre_processing.flow_from_dataframe(
    dataframe=Test,
    x_col='Filepath',
    y_col='Labels',
    target_size=(48, 48),
    class_mode='categorical',
    batch_size=32,
    color_mode='grayscale',
    shuffle=False,
    seed=42
)

Test_pre_processed = test_pre_processing.flow_from_dataframe(
    dataframe=Test,
    x_col='Filepath',
    y_col='Labels',
    target_size=(48, 48),
    color_mode='grayscale',
    class_mode='categorical',
    batch_size=32,
    shuffle=False
)

Found 2441 validated image filenames belonging to 7 classes.
Found 611 validated image filenames belonging to 7 classes.
Found 611 validated image filenames belonging to 7 classes.


In [19]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal", input_shape=(48, 48, 1)),
    layers.RandomRotation(0.3),
    layers.RandomZoom(0.45),
])

/opt/homebrew/lib/python3.9/site-packages/keras/src/layers/preprocessing/tf_data_layer.py:19: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [21]:
model = keras.models.load_model('emotion_model_1000.keras')

In [22]:
for layer in model.layers[:-3]:  # Assuming last few layers to fine-tune
    layer.trainable = False


In [23]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


In [24]:
my_callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=10, mode='auto'),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-5)
]

In [26]:
history = model.fit(
    train_pre_processed,
    validation_data=Validation_pre_processed,
    epochs=500
)


Epoch 1/500
77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.4677 - loss: 1.4603 - val_accuracy: 0.5106 - val_loss: 1.3280
Epoch 2/500
77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4695 - loss: 1.4263 - val_accuracy: 0.5123 - val_loss: 1.3249
Epoch 3/500
77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.4620 - loss: 1.4244 - val_accuracy: 0.5221 - val_loss: 1.3335
Epoch 4/500
77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.4450 - loss: 1.4982 - val_accuracy: 0.5074 - val_loss: 1.3381
Epoch 5/500
77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4612 - loss: 1.4422 - val_accuracy: 0.5025 - val_loss: 1.3293
Epoch 6/500
77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.4602 - loss: 1.4441 - val_accuracy: 0.5057 - val_loss: 1.3264
Epoch 7/500
77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4745 - loss: 1.4354 - val_accuracy: 0.5106 - val_loss: 1.3299
Epoch 8/500
77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4754 - loss: 1.4265 - val_accuracy: 0.

In [ ]:
model.save('fine_tuned_emotion_model.keras')